In [15]:
print("Installing lightgbm...")
%pip install lightgbm
import lightgbm as lgb
print("Installed & Imported!")

Installing lightgbm...
Note: you may need to restart the kernel to use updated packages.
Installed & Imported!


In [16]:
import numpy as np
import pandas as pd
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

In [17]:
# july_test = pd.read_parquet('clean_july_test.parquet')
# may_jun_train = pd.read_parquet('clean_may_jun_training.parquet')

In [18]:
TRAIN_PATH = 'clean_may_jun_training.parquet'
TEST_PATH = 'clean_july_test.parquet'
TARGET = "is_fraud"
cols_to_drop_from_traintest = ["confirmation_number","address_line_1","first_name","last_name","zip",
 "check_in_date", "booking_timestamp", "check_out_date", "cancellation_date", "site_id", "site_name",
 "member_number", "member_enrollment_date", "email", "phone_number", "city", "qualification_code", "data_month", "split",
 "is_fraud", "points_earned"
 ]

In [19]:
train_raw = pd.read_parquet(TRAIN_PATH)
test_raw = pd.read_parquet(TEST_PATH)
train_raw["is_test"] = False
test_raw["is_test"] = True

In [20]:
#combines the two tables
raw = pd.concat([train_raw, test_raw], ignore_index=True)
print(f"Train rows: {len(train_raw):,}   Fraud rate: {train_raw[TARGET].mean():.2%}")
print(f"Test rows:  {len(test_raw):,}   Fraud rate: {test_raw[TARGET].mean():.2%}")

Train rows: 6,400,000   Fraud rate: 0.05%
Test rows:  3,200,000   Fraud rate: 0.05%


In [21]:
#raw.columns

**Building the features**

In [22]:
X = raw.drop(columns=cols_to_drop_from_traintest) #pd.DataFrame(index=raw.index)

In [23]:
X["lead_time_days"] = (raw["check_in_date"] - raw["booking_timestamp"]).dt.total_seconds() / 86400
X["nights"] = (raw["check_out_date"] - raw["check_in_date"]).dt.days
#X["room_revenue"] = raw["room_revenue"]
#X["revenue_per_night"] = raw["room_revenue"] / X["nights"].clip(lower=1)
#X["points_redeemed"] = raw["points_redeemed"]
X["booking_hour"] = raw["booking_timestamp"].dt.hour
X["booking_dow"] = raw["booking_timestamp"].dt.dayofweek
#X["account_age"] = raw["account_age"]
X["email_digits"] = raw["email"].str.split("@").str[0].str.count(r"\d")
X["email_domain"] = raw["email"].str.split("@").str[1]
top_domains = X["email_domain"].value_counts().head(20).index
X["email_domain"] = X["email_domain"].where(X["email_domain"].isin(top_domains), "other")

for col in ["member_number", "email", "phone_number", "address_line_1"]:
    X[f"{col}_prior_bookings"] = raw.groupby(col, dropna=False).cumcount()

cat_cols = ["rate_code", "booking_channel", "site_brand", "member_enrollment_source", "member_language_preference", "state", "country"]
for col in cat_cols:
    X[col] = raw[col]
cat_cols.append("email_domain")
X[cat_cols] = X[cat_cols].astype("category")
X.head()
#X = X.drop(columns=[c for c in X.columns if c.endswith("_prior_bookings")])

,rate_code,room_revenue,points_redeemed,booking_channel,site_brand,member_enrollment_source,member_language_preference,state,country,has_stay_row,...,lead_time_days,nights,booking_hour,booking_dow,email_digits,email_domain,member_number_prior_bookings,email_prior_bookings,phone_number_prior_bookings,address_line_1_prior_bookings
0,PKG,639.63,NaN,Property Direct,Crescent Crown,NaN,NaN,PA,US,True,...,89.514398,3,11,3,5,proton.me,0,0,0,0
1,RROD,720.54,NaN,Property Direct,Solara Grand,NaN,NaN,CA,US,True,...,112.045868,2,22,4,5,outlook.com,1,0,0,0
2,FLEX,805.99,NaN,OTA,Harborlight Hotels,NaN,NaN,PA,US,True,...,0.500752,5,11,6,5,icloud.com,2,0,0,0
3,RROD,406.96,NaN,Property Direct,Crescent Crown,NaN,NaN,PA,US,True,...,124.109560,2,21,4,5,gmail.com,3,0,0,0
4,RROD,182.54,NaN,Voice,Summit Stay,ONLINE_WEB,en_US,TX,US,True,...,38.376319,1,14,4,5,outlook.com,0,0,0,0


In [24]:
test = raw["is_test"].to_numpy()
y = raw[TARGET].astype(int).to_numpy()
print(f"Training period: {raw.loc[~test, 'booking_timestamp'].min():%Y-%m-%d} to {raw.loc[~test, 'booking_timestamp'].max():%Y-%m-%d}")
print(f"Test period:     {raw.loc[test, 'booking_timestamp'].min():%Y-%m-%d} to {raw.loc[test, 'booking_timestamp'].max():%Y-%m-%d}")

Training period: 2025-10-27 to 2026-06-28
Test period:     2025-12-27 to 2026-07-28


In [25]:
#training the light gbm model
print("Training...")
model = lgb.LGBMClassifier(
       n_estimators=200, learning_rate=0.05,
       num_leaves=7, min_child_samples=5,
       scale_pos_weight=min((~test).sum() / max(y[~test].sum(), 1), 50),
       subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
       random_state=42, verbose=-1,
   )
model.fit(X[~test], y[~test])

Training...


,boosting_type,'gbdt'
,num_leaves,7
,max_depth,-1
,learning_rate,0.05
,n_estimators,200
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,5


In [26]:
risk_score = model.predict_proba(X[test])[:, 1]
results = pd.DataFrame({
    "confirmation_number": raw.loc[test, "confirmation_number"].to_numpy(),
    "risk_score": risk_score,
})
results["risk_band"] = pd.qcut(results["risk_score"].rank(method="first"), [0, .8, .95, .99, 1], labels=["low", "medium", "high", "critical"])
results.sort_values("risk_score", ascending=False).head(10)

,confirmation_number,risk_score,risk_band
2704374,HTL2601167328,1.0,critical
191147,HTL2600843066,1.0,critical
2594442,HTL2600734477,1.0,critical
3114329,HTL2602554974,1.0,critical
667226,HTL2602275183,1.0,critical
1279664,HTL2601819765,1.0,critical
711794,HTL2602387813,1.0,critical
1849000,HTL2600702211,1.0,critical
1528965,HTL2602822958,1.0,critical
844211,HTL2600428192,1.0,critical


In [27]:
y_test = y[test]
print(f"Test fraud rate: {y_test.mean():.2%}")
print(f"ROC-AUC: {roc_auc_score(y_test, risk_score):.3f}   (0.5 = random, 1.0 = perfect)")
print(f"PR-AUC:  {average_precision_score(y_test, risk_score):.3f}   (random = {y_test.mean():.3f})")
for frac in [0.01, 0.05, 0.10]:
    k = int(len(y_test) * frac)
    caught = y_test[np.argsort(-risk_score)[:k]].sum()
    print(f"Top {frac:.0%}: {caught / k:.1%} are fraud, catching {caught / y_test.sum():.1%} of all fraud")

Test fraud rate: 0.05%
ROC-AUC: 0.647   (0.5 = random, 1.0 = perfect)
PR-AUC:  0.023   (random = 0.001)
Top 1%: 1.5% are fraud, catching 29.6% of all fraud
Top 5%: 0.3% are fraud, catching 29.6% of all fraud
Top 10%: 0.1% are fraud, catching 29.6% of all fraud


In [28]:
pd.Series(model.booster_.feature_importance("gain"), index=X.columns).sort_values(ascending=False).head(10)

address_line_1_prior_bookings    2.389405e+13
room_revenue                     4.577441e+12
account_age                      3.275581e+12
phone_number_prior_bookings      9.842160e+11
email_prior_bookings             7.889995e+11
lead_time_days                   7.669319e+11
booking_hour                     5.770963e+11
member_number_prior_bookings     2.027198e+11
nights                           1.952887e+11
booking_dow                      4.575984e+10
dtype: float64